In [2]:
read_file_tool = {
    "name": "read_file",
    "description": (
        "Reads a .txt file at the given path and returns its full contents as a string."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {
                "type": "string",
                "description": "Path to the .txt file, such as 'notes.txt'."
            }
        },
        "required": ["file_path"]
    }
}

In [3]:
from langchain_core.tools import tool
import pandas as pd

@tool
def read_csv(file_path: str) -> str:
    """Read a CSV file from the local filesystem and return its contents as text."""
    df = pd.read_csv(file_path)
    return df.to_string(index=False)


In [4]:
products = pd.DataFrame({
    "product": ["Laptop", "Headphones", "Keyboard", "Mouse"],
    "price": [120000, 8500, 4500, 2500],
    "category": ["Computer", "Audio", "Accessories", "Accessories"]
})

products.to_csv("products.csv", index=False)

print(products)

      product   price     category
0      Laptop  120000     Computer
1  Headphones    8500        Audio
2    Keyboard    4500  Accessories
3       Mouse    2500  Accessories


# **TASK 1**

### Core LangGraph Building Blocks

**StateGraph**  
`StateGraph` is the main graph builder in LangGraph. It defines the workflow, the shared state schema, the nodes that perform work, and the transitions between those nodes.

**Nodes**  
Nodes are Python functions that perform individual workflow steps. Each node receives the current graph state and returns updates that are merged back into that state.

**Edges**  
Edges define the fixed transitions between nodes. For example, an edge from `plan` to `retrieve` means that `retrieve` executes after `plan`.

**Conditional Edges**  
Conditional edges dynamically select the next node based on the current state. They are useful for branching, retries, validation, and self-correction loops.

**Shared State**  
The State object is the data shared across the complete workflow. Every node can read values from the state and return updates to those values. This makes workflow execution explicit and stateful.

### State Schema

In [ ]:
from typing import TypedDict

class ResearchState(TypedDict):
    question: str
    plan: str
    retrieved_data: str
    draft: str
    critique: str
    quality_score: int
    retry_count: int
    max_retries: int
    approved: bool | None
    final_answer: str

### Graph Diagram

                    ┌──────────────┐
                    │     START    │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │     PLAN     │
                    │ Create plan  │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │   RETRIEVE   │
                    │ Get relevant │
                    │    data      │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │   GENERATE   │
                    │ Draft answer │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │   CRITIQUE   │
                    │ Check quality│
                    └──────┬───────┘
                           │
                 ┌─────────┴─────────┐
                 │                   │
          Quality < threshold   Quality ≥ threshold
                 │                   │
                 ▼                   ▼
          ┌──────────────┐    ┌──────────────┐
          │   GENERATE   │    │    HUMAN     │
          │   (REVISE)   │    │   APPROVAL   │
          └──────┬───────┘    └──────┬───────┘
                 │                   │
                 └───► CRITIQUE      │
                                     ▼
                              ┌──────────────┐
                              │    FINISH    │
                              │ Final answer │
                              └──────────────┘

# **TASK 2**

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict):
    input: str
    plan: str
    retrieved_data: str
    draft: str
    final_answer: str

In [9]:
import pandas as pd

#Plan Node
def plan_node(state: AgentState):
    plan = (
        "1. Understand the user's request\n"
        "2. Retrieve the required product information\n"
        "3. Generate an answer\n"
        "4. Format the final response"
    )

    print("\n--- PLAN NODE ---")
    print(plan)

    return {
        "plan": plan
    }


#Retrive Node
def retrieve_node(state: AgentState):
    df = pd.read_csv("products.csv")

    retrieved_data = df.to_string(index=False)

    print("\n--- RETRIEVE NODE ---")
    print(retrieved_data)

    return {
        "retrieved_data": retrieved_data
    }

#Genrate Node
def generate_node(state: AgentState):
    print("\n--- GENERATE NODE ---")

    df = pd.read_csv("products.csv")

    laptop = df[df["product"].str.lower() == "laptop"]

    if not laptop.empty:
        price = laptop.iloc[0]["price"]
        draft = f"The price of the Laptop is ₦{price:,.0f}."
    else:
        draft = "Laptop was not found in the product data."

    print("Draft:", draft)

    return {
        "draft": draft
    }

#Format Node
def format_node(state: AgentState):
    final_answer = (
        "Final Response:\n\n"
        + state["draft"]
    )

    print("\n--- FORMAT NODE ---")
    print(final_answer)

    return {
        "final_answer": final_answer
    }

### Linear Graph

In [10]:
workflow = StateGraph(AgentState)

workflow.add_node("plan", plan_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("format", format_node)

workflow.add_edge(START, "plan")
workflow.add_edge("plan", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "format")
workflow.add_edge("format", END)

graph = workflow.compile()

print("Linear LangGraph compiled successfully.")

Linear LangGraph compiled successfully.


### Test

In [11]:
result = graph.invoke({
    "input": "Find the price of the Laptop."
})


--- PLAN NODE ---
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Format the final response

--- RETRIEVE NODE ---
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

--- GENERATE NODE ---
Draft: The price of the Laptop is ₦120,000.

--- FORMAT NODE ---
Final Response:

The price of the Laptop is ₦120,000.


### State Update

In [12]:
print("=== STATE UPDATES ===")

for state in graph.stream(
    {"input": "Find the price of the Laptop."},
    stream_mode="values"
):
    print("\n" + "=" * 50)
    print("CURRENT STATE")
    print("=" * 50)

    for key, value in state.items():
        print(f"\n{key}:")
        print(value)

=== STATE UPDATES ===

CURRENT STATE

input:
Find the price of the Laptop.

--- PLAN NODE ---
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Format the final response

CURRENT STATE

input:
Find the price of the Laptop.

plan:
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Format the final response

--- RETRIEVE NODE ---
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

CURRENT STATE

input:
Find the price of the Laptop.

plan:
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Format the final response

retrieved_data:
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

--- GENERATE NODE ---
Draft: The price of the Laptop is ₦1

# **TASK 3**

### Adding Conditional edge(Critic Node)

In [13]:
from typing import TypedDict

class AgentState(TypedDict):
    input: str
    plan: str
    retrieved_data: str
    draft: str
    critique: str
    quality_score: int
    retry_count: int
    max_retries: int
    final_answer: str

In [14]:
def plan_node(state: AgentState):
    plan = (
        "1. Understand the user's request\n"
        "2. Retrieve the required product information\n"
        "3. Generate an answer\n"
        "4. Critique the answer\n"
        "5. Revise if necessary\n"
        "6. Return the final answer"
    )

    print("\n--- PLAN NODE ---")
    print(plan)

    return {"plan": plan}

def retrieve_node(state: AgentState):
    retrieved_data = read_csv.invoke({
        "file_path": "products.csv"
    })

    print("\n--- RETRIEVE NODE ---")
    print(retrieved_data)

    return {"retrieved_data": retrieved_data}

def generate_node(state: AgentState):
    print("\n--- GENERATE NODE ---")

    retry = state["retry_count"]

    if retry == 0:
        draft = "The price is 120000."
    else:
        draft = (
            "The price of the Laptop is ₦120,000. "
            "The Laptop is listed in the Computer category."
        )

    print(f"Generation pass: {retry + 1}")
    print("Draft:", draft)

    return {"draft": draft}

#Critique Node
def critique_node(state: AgentState):
    print("\n--- CRITIQUE NODE ---")

    draft = state["draft"]

    # Simple quality check for demonstration
    if len(draft.split()) < 8:
        quality_score = 4
        critique = (
            "The answer is too brief. "
            "It should provide more useful information."
        )
    else:
        quality_score = 9
        critique = "The answer is clear, complete, and acceptable."

    new_retry_count = state["retry_count"] + 1

    print("Quality Score:", quality_score)
    print("Critique:", critique)
    print("Retry Count:", new_retry_count)

    return {
        "quality_score": quality_score,
        "critique": critique,
        "retry_count": new_retry_count
    }


#Conditional Router
def critique_router(state: AgentState):
    if state["quality_score"] >= 8:
        print("Routing decision: Quality acceptable → FINISH")
        return "finish"

    if state["retry_count"] >= state["max_retries"]:
        print("Routing decision: Maximum retries reached → FINISH")
        return "finish"

    print("Routing decision: Quality too low → GENERATE again")
    return "generate"

def finish_node(state: AgentState):
    print("\n--- FINISH NODE ---")

    final_answer = state["draft"]

    print("Final Answer:", final_answer)

    return {
        "final_answer": final_answer
    }

### Updating Graph

In [15]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

workflow.add_node("plan", plan_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("critique", critique_node)
workflow.add_node("finish", finish_node)

# Linear edges
workflow.add_edge(START, "plan")
workflow.add_edge("plan", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "critique")

# Conditional edge
workflow.add_conditional_edges(
    "critique",
    critique_router,
    {
        "generate": "generate",
        "finish": "finish"
    }
)

workflow.add_edge("finish", END)

cycle_graph = workflow.compile()

### Test

In [16]:
result = cycle_graph.invoke({
    "input": "Find the price of the Laptop.",
    "retry_count": 0,
    "max_retries": 3
})

print("\n=== FINAL STATE ===")

for key, value in result.items():
    print(f"\n{key}:")
    print(value)


--- PLAN NODE ---
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Critique the answer
5. Revise if necessary
6. Return the final answer

--- RETRIEVE NODE ---
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

--- GENERATE NODE ---
Generation pass: 1
Draft: The price is 120000.

--- CRITIQUE NODE ---
Quality Score: 4
Critique: The answer is too brief. It should provide more useful information.
Retry Count: 1
Routing decision: Quality too low → GENERATE again

--- GENERATE NODE ---
Generation pass: 2
Draft: The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.

--- CRITIQUE NODE ---
Quality Score: 9
Critique: The answer is clear, complete, and acceptable.
Retry Count: 2
Routing decision: Quality acceptable → FINISH

--- FINISH NODE ---
Final Answer: The price of the Laptop is ₦120,000. The Laptop is list

In a plain AgentExecutor, loop-back behavior is mainly controlled by the agent's internal reasoning and tool-calling process, making explicit retry paths and termination conditions less transparent. LangGraph makes this pattern natural because conditional edges can explicitly route execution from a critique node back to a generation node while maintaining shared state and a retry counter. This provides clear control over cycles and prevents infinite loops.

# **TASK 4**

In [67]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class AgentState(TypedDict):
    input: str
    plan: str
    retrieved_data: str
    draft: str
    critique: str
    quality_score: int
    retry_count: int
    max_retries: int
    approved: bool
    action_result: str
    final_answer: str

In [68]:
def plan_node(state: AgentState):
    plan = (
        "1. Understand the user's request\n"
        "2. Retrieve the required product information\n"
        "3. Generate an answer\n"
        "4. Critique the answer\n"
        "5. Revise if necessary\n"
        "6. Return the final answer"
    )

    print("\n--- PLAN NODE ---")
    print(plan)

    return {"plan": plan}

def retrieve_node(state: AgentState):
    retrieved_data = read_csv.invoke({
        "file_path": "products.csv"
    })

    print("\n--- RETRIEVE NODE ---")
    print(retrieved_data)

    return {"retrieved_data": retrieved_data}

def generate_node(state: AgentState):
    print("\n--- GENERATE NODE ---")

    retry = state["retry_count"]

    if retry == 0:
        draft = "The price is 120000."
    else:
        draft = (
            "The price of the Laptop is ₦120,000. "
            "The Laptop is listed in the Computer category."
        )

    print(f"Generation pass: {retry + 1}")
    print("Draft:", draft)

    return {"draft": draft}

#Critique Node
def critique_node(state: AgentState):
    print("\n--- CRITIQUE NODE ---")

    draft = state["draft"]

    # Simple quality check for demonstration
    if len(draft.split()) < 8:
        quality_score = 4
        critique = (
            "The answer is too brief. "
            "It should provide more useful information."
        )
    else:
        quality_score = 9
        critique = "The answer is clear, complete, and acceptable."

    new_retry_count = state["retry_count"] + 1

    print("Quality Score:", quality_score)
    print("Critique:", critique)
    print("Retry Count:", new_retry_count)

    return {
        "quality_score": quality_score,
        "critique": critique,
        "retry_count": new_retry_count
    }


#Conditional Router
def critique_router(state: AgentState):
    if state["quality_score"] >= 8:
        print("Routing decision: Quality acceptable → HUMAN APPROVAL")
        return "human_approval"

    if state["retry_count"] >= state["max_retries"]:
        print("Routing decision: Maximum retries reached → HUMAN APPROVAL")
        return "human_approval"

    print("Routing decision: Quality too low → GENERATE again")
    return "generate"

#Human approval
def human_approval_node(state: AgentState):
    print("\n--- HUMAN APPROVAL ---")
    print("Human approval required before risky action.")
    print("Action: Place an order for Laptop")
    print("Price: ₨120,000")

    while True:
        response = input(
            "\nApprove this action? Enter True or False: "
        ).strip().lower()

        if response == "true":
            approved = True
            break

        elif response == "false":
            approved = False
            break

        else:
            print("Invalid input. Please enter True or False.")

    print(
        "Human response:",
        "APPROVED" if approved else "REJECTED"
    )

    return {
        "approved": approved
    }


# Approval Router
def approval_router(state: AgentState):
    if state["approved"]:
        print("Routing decision: APPROVED → RISKY ACTION")
        return "risky_action"

    print("Routing decision: REJECTED → FINISH")
    return "finish"


# Risky Action
def risky_action_node(state: AgentState):
    print("\n--- RISKY ACTION ---")

    action_result = (
        "Purchase approved. "
        "Simulated order for Laptop at ₨120,000 was executed."
    )

    print(action_result)

    return {
        "action_result": action_result
    }


def finish_node(state: AgentState):
    print("\n--- FINISH NODE ---")

    if state["approved"]:
        final_answer = state["action_result"]
    else:
        final_answer = "Purchase action rejected by the human reviewer."

    print("Final Answer:", final_answer)

    return {
        "final_answer": final_answer
    }

### Updated Graph

In [69]:
workflow = StateGraph(AgentState)

workflow.add_node("plan", plan_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("critique", critique_node)
workflow.add_node("human_approval", human_approval_node)
workflow.add_node("risky_action", risky_action_node)
workflow.add_node("finish", finish_node)

workflow.add_edge(START, "plan")
workflow.add_edge("plan", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "critique")

workflow.add_conditional_edges(
    "critique",
    critique_router,
    {
        "generate": "generate",
        "human_approval": "human_approval"
    }
)

workflow.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "risky_action": "risky_action",
        "finish": "finish"
    }
)

workflow.add_edge("risky_action", "finish")
workflow.add_edge("finish", END)

In [70]:
hitl_graph = workflow.compile(
    interrupt_before=["risky_action"]
)

### Run Graph

In [72]:
result = hitl_graph.invoke({
    "input": "Place an order for the Laptop.",
    "retry_count": 0,
    "max_retries": 2,
    "approved": False,
    "action_result": "",
    "final_answer": ""
})

print("\n=== FINAL RESULT ===")
print(result)


--- PLAN NODE ---
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Critique the answer
5. Revise if necessary
6. Return the final answer

--- RETRIEVE NODE ---
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

--- GENERATE NODE ---
Generation pass: 1
Draft: The price is 120000.

--- CRITIQUE NODE ---
Quality Score: 4
Critique: The answer is too brief. It should provide more useful information.
Retry Count: 1
Routing decision: Quality too low → GENERATE again

--- GENERATE NODE ---
Generation pass: 2
Draft: The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.

--- CRITIQUE NODE ---
Quality Score: 9
Critique: The answer is clear, complete, and acceptable.
Retry Count: 2
Routing decision: Quality acceptable → HUMAN APPROVAL

--- HUMAN APPROVAL ---
Human approval required before risky action.
Action: Place

For a real-world agent, human-in-the-loop (HITL) should be required when an agent's action could cause significant harm, financial loss, legal consequences, privacy issues, safety risks, or irreversible changes. Examples include making purchases, transferring money, deleting important data, sending legally significant communications, making medical decisions, or deploying changes to production systems. HITL is also useful when the agent has low confidence or encounters ambiguous or conflicting information.

Full autonomy is acceptable for tasks that are low-risk, well-defined, reversible, and easy for a user to verify or correct. Examples include summarizing documents, organizing information, classifying data, generating drafts, answering routine questions, or performing calculations.

Key principle: the level of autonomy should be proportional to the potential risk. High-impact or irreversible actions should have human oversight, while routine and low-risk tasks can safely operate autonomously.

# **TASK 5**

### Adding Check Pointer

In [73]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

checkpointer = MemorySaver()

persistent_graph = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["risky_action"]
)

### Adding Thread

In [74]:
config = {
    "configurable": {
        "thread_id": "task5-order-001"
    }
}

In [75]:
paused_state = persistent_graph.invoke(
    {
        "input": "Place an order for the Laptop.",
        "retry_count": 0,
        "max_retries": 2,
        "approved": False,
        "action_result": "",
        "final_answer": ""
    },
    config=config
)

print("\n=== PAUSED STATE ===")
print(paused_state)


--- PLAN NODE ---
1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Critique the answer
5. Revise if necessary
6. Return the final answer

--- RETRIEVE NODE ---
   product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories

--- GENERATE NODE ---
Generation pass: 1
Draft: The price is 120000.

--- CRITIQUE NODE ---
Quality Score: 4
Critique: The answer is too brief. It should provide more useful information.
Retry Count: 1
Routing decision: Quality too low → GENERATE again

--- GENERATE NODE ---
Generation pass: 2
Draft: The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.

--- CRITIQUE NODE ---
Quality Score: 9
Critique: The answer is clear, complete, and acceptable.
Retry Count: 2
Routing decision: Quality acceptable → HUMAN APPROVAL

--- HUMAN APPROVAL ---
Human approval required before risky action.
Action: Place

In [76]:
current_state = persistent_graph.get_state(config)

print("\n=== SAVED STATE ===")
print(current_state)


=== SAVED STATE ===
StateSnapshot(values={'input': 'Place an order for the Laptop.', 'plan': "1. Understand the user's request\n2. Retrieve the required product information\n3. Generate an answer\n4. Critique the answer\n5. Revise if necessary\n6. Return the final answer", 'retrieved_data': '   product  price    category\n    Laptop 120000    Computer\nHeadphones   8500       Audio\n  Keyboard   4500 Accessories\n     Mouse   2500 Accessories', 'draft': 'The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.', 'critique': 'The answer is clear, complete, and acceptable.', 'quality_score': 9, 'retry_count': 2, 'max_retries': 2, 'approved': False, 'action_result': '', 'final_answer': 'Purchase action rejected by the human reviewer.'}, next=(), config={'configurable': {'thread_id': 'task5-order-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac2dd-4f32-6581-8008-b58410e80ed4'}}, metadata={'source': 'loop', 'step': 8, 'parents': {}}, created_at='2026-09-09T09:0

### Resuming Graph

In [77]:
resumed_state = persistent_graph.invoke(
    Command(resume=True),
    config=config
)

print("\n=== RESUMED STATE ===")
print(resumed_state)


=== RESUMED STATE ===
{'input': 'Place an order for the Laptop.', 'plan': "1. Understand the user's request\n2. Retrieve the required product information\n3. Generate an answer\n4. Critique the answer\n5. Revise if necessary\n6. Return the final answer", 'retrieved_data': '   product  price    category\n    Laptop 120000    Computer\nHeadphones   8500       Audio\n  Keyboard   4500 Accessories\n     Mouse   2500 Accessories', 'draft': 'The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.', 'critique': 'The answer is clear, complete, and acceptable.', 'quality_score': 9, 'retry_count': 2, 'max_retries': 2, 'approved': False, 'action_result': '', 'final_answer': 'Purchase action rejected by the human reviewer.'}


### State History

In [79]:
history = list(
    persistent_graph.get_state_history(config)
)

print("\n=== STATE HISTORY ===")
print(f"Number of checkpoints: {len(history)}")

for i, snapshot in enumerate(history):
    print("\n" + "=" * 60)
    print(f"CHECKPOINT {i}")
    print("=" * 60)

    print("Next:", snapshot.next)

    print("\nState:")

    for key, value in snapshot.values.items():
        print(f"  {key}: {value}")

for i, snapshot in enumerate(history):
    print(
        f"Checkpoint {i}:",
        snapshot.config["configurable"].get("checkpoint_id"),
        "| Next:",
        snapshot.next
    )


checkpoint = history[0]

print("\n=== HISTORICAL SNAPSHOT ===")

for key, value in checkpoint.values.items():
    print(f"{key}: {value}")


=== STATE HISTORY ===
Number of checkpoints: 10

CHECKPOINT 0
Next: ()

State:
  input: Place an order for the Laptop.
  plan: 1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Critique the answer
5. Revise if necessary
6. Return the final answer
  retrieved_data:    product  price    category
    Laptop 120000    Computer
Headphones   8500       Audio
  Keyboard   4500 Accessories
     Mouse   2500 Accessories
  draft: The price of the Laptop is ₦120,000. The Laptop is listed in the Computer category.
  critique: The answer is clear, complete, and acceptable.
  quality_score: 9
  retry_count: 2
  max_retries: 2
  approved: False
  action_result: 
  final_answer: Purchase action rejected by the human reviewer.

CHECKPOINT 1
Next: ('finish',)

State:
  input: Place an order for the Laptop.
  plan: 1. Understand the user's request
2. Retrieve the required product information
3. Generate an answer
4. Critique the answer
5. Revise if ne

### Final State

In [80]:
final_state = persistent_graph.get_state(config)

print("\n=== FINAL SAVED STATE ===")
print("Approved:", final_state.values["approved"])
print("Action Result:", final_state.values["action_result"])
print("Final Answer:", final_state.values["final_answer"])


=== FINAL SAVED STATE ===
Approved: False
Action Result: 
Final Answer: Purchase action rejected by the human reviewer.


### LangChain AgentExecutor vs. LangGraph

`AgentExecutor` is a good choice for relatively simple agent applications where an LLM decides which tools to call and the workflow is mostly linear. It is useful when the main requirement is straightforward tool use without complex state management, explicit branching, human approval steps, or long-running workflows.

`LangGraph` is better suited for complex, stateful agent systems that require explicit control over workflow execution. It provides nodes, conditional edges, cycles, persistence, checkpointing, human-in-the-loop interruptions, and state-history debugging. In a real project, I would use `AgentExecutor` for a simple tool-calling assistant and LangGraph when the application requires reliable multi-step workflows, retries, persistence, human approval, or recovery from interrupted execution.

**In short:** `AgentExecutor` is convenient for simpler agent loops, while LangGraph is preferable when the workflow itself needs to be explicitly designed, controlled, persisted, and debugged.
